# AeroPure — Week 2: Data Cleaning, AQI Proxy & Feature Engineering

### 1. Overview
In this notebook, we:
1. Clean the dataset (parsing timestamps, chronological sorting, imputation).
2. Calculate a **Pollutant-Based Air Quality Index Proxy** using standard piecewise linear interpolation for CO(GT), NO2(GT), and C6H6(GT).
3. Construct leakage-safe targets: `next_day_air_quality_index` (regression) and `hazardous_air_day` (classification).
4. Extract temporal, lag ($t-1, t-2, t-3, t-24, t-48$), and rolling window (6h, 12h, 24h) features.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath(".."))
from src.preprocessing import load_raw_archive1, clean_dataset
from src.aqi import calculate_pollutant_index_proxy, create_targets
from src.feature_engineering import build_feature_pipeline, prepare_time_series_splits

DATA_PATH = os.path.join("..", "data", "AirQuality.csv")
if os.path.exists(DATA_PATH):
    df_raw = load_raw_archive1(DATA_PATH)
    df_clean, clean_log = clean_dataset(df_raw)
    print(f"Cleaned dataset: {len(df_clean)} records")
    print("Data cleaning log:", clean_log)


### 2. Pollutant-Based AQI Proxy Calculation
Using standard breakpoint sub-indices:
$$I_p = \frac{I_{hi} - I_{lo}}{BP_{hi} - BP_{lo}} (C_p - BP_{lo}) + I_{lo}$$
$$current\_air\_quality\_index = \max(I_{CO}, I_{NO2}, I_{C6H6})$$


In [ ]:
if "df_clean" in locals():
    df_aqi = calculate_pollutant_index_proxy(df_clean)
    df_targets = create_targets(df_aqi, lead_time_hours=24, hazard_threshold=180.0)
    print("Calculated AQI Proxy Summary:")
    display(df_targets[['current_air_quality_index', 'dominant_pollutant', 'next_day_air_quality_index', 'hazardous_air_day']].describe())


### 3. Leakage-Safe Feature Engineering
Extracting temporal features, lags, rolling averages, and interaction terms.
**Critical Rule**: Features at time $t$ use only historical data up to $t$.


In [ ]:
if "df_targets" in locals():
    feature_df = build_feature_pipeline(df_targets, lead_time_hours=24, hazard_threshold=180.0)
    print(f"Constructed feature matrix: {feature_df.shape[0]} rows, {feature_df.shape[1]} columns")
    display(feature_df.head(3))


### 4. Chronological Train/Test Split
Splitting into Train (80%) and Test (20%) chronologically. Scaler is fit strictly on Train partition.


In [ ]:
if "feature_df" in locals():
    (
        X_train, X_test,
        y_train_reg, y_test_reg,
        y_train_clf, y_test_clf,
        scaler, feature_cols
    ) = prepare_time_series_splits(feature_df, train_ratio=0.80)
    print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")
    print(f"Input features count: {len(feature_cols)}")


### Week 2 Summary
- Implemented transparent multi-pollutant AQI proxy with standard sub-index breakpoints.
- Produced regression target `next_day_air_quality_index` and classification target `hazardous_air_day`.
- Created 113 engineered features with zero lookahead bias.
- Established strict chronological 80/20 train/test split.
